In [1]:
# Cell 1 — Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
print("✅ All NLP libraries loaded!")

PyTorch version: 2.11.0+cpu
Device: cpu
✅ All NLP libraries loaded!


In [2]:
# Cell 2 — Load IMDb Dataset
print("⏳ Loading IMDb dataset... (may take 1-2 mins)")

dataset = load_dataset("imdb")

# Use smaller subset for CPU training
train_data = dataset['train'].shuffle(seed=42).select(range(2000))
val_data = dataset['test'].shuffle(seed=42).select(range(500))

print(f"✅ Dataset loaded!")
print(f"Training samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")
print(f"\nSample review:")
print(f"Text: {train_data[0]['text'][:200]}...")
print(f"Label: {'Positive' if train_data[0]['label'] == 1 else 'Negative'}")

⏳ Loading IMDb dataset... (may take 1-2 mins)


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

✅ Dataset loaded!
Training samples: 2000
Validation samples: 500

Sample review:
Text: There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. F...
Label: Positive


In [3]:
# Cell 3 — Tokenize Data
print("⏳ Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_function(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        max_length=256,
        padding='max_length'
    )

print("⏳ Tokenizing dataset...")
train_tok = train_data.map(tokenize_function, batched=True)
val_tok = val_data.map(tokenize_function, batched=True)

# Rename label column
train_tok = train_tok.rename_column("label", "labels")
val_tok = val_tok.rename_column("label", "labels")

# Set format for PyTorch
train_tok.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
val_tok.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

print(f"✅ Tokenization done!")
print(f"Train tokens shape: {train_tok[0]['input_ids'].shape}")
print(f"Sample labels: {train_tok[0]['labels']}")

⏳ Loading tokenizer...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

⏳ Tokenizing dataset...


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

✅ Tokenization done!
Train tokens shape: torch.Size([256])
Sample labels: 1


In [4]:
# Cell 4 — Load DistilBERT Model
print("⏳ Loading DistilBERT model...")

model_nlp = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

# Count parameters
total_params = sum(p.numel() for p in model_nlp.parameters())
trainable_params = sum(p.numel() for p in model_nlp.parameters() if p.requires_grad)

print(f"✅ DistilBERT loaded!")
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\nModel config:")
print(f"Hidden size: {model_nlp.config.hidden_size}")
print(f"Num layers:  {model_nlp.config.n_layers}")
print(f"Num heads:   {model_nlp.config.n_heads}")

⏳ Loading DistilBERT model...


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ DistilBERT loaded!
Total parameters:     66,955,010
Trainable parameters: 66,955,010

Model config:
Hidden size: 768
Num layers:  6
Num heads:   12


In [6]:
# Cell 5 — Fine-tune DistilBERT
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'f1': f1_score(labels, predictions)
    }

training_args = TrainingArguments(
    output_dir='../models/saved/bert_sentiment',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    logging_steps=50,
    warmup_steps=100,
    weight_decay=0.01,
    report_to='none'
)

trainer = Trainer(
    model=model_nlp,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    compute_metrics=compute_metrics
)

print("🚀 Starting DistilBERT fine-tuning...")
print("⚠️  This takes 15-20 mins on CPU — don't close VS Code!\n")
trainer.train()
print("\n✅ Fine-tuning complete!")

🚀 Starting DistilBERT fine-tuning...
⚠️  This takes 15-20 mins on CPU — don't close VS Code!



Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.310677,0.345248,0.852000,0.856031
2,0.247630,0.387149,0.894000,0.891616
3,0.096918,0.437122,0.880000,0.882353


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Fine-tuning complete!


In [7]:
# Cell 6 — Evaluate Model
print("🔍 Evaluating model...")

# Get evaluation results
eval_results = trainer.evaluate()
print(f"\n✅ Evaluation Results:")
print(f"Accuracy: {eval_results['eval_accuracy']:.4f}")
print(f"F1 Score: {eval_results['eval_f1']:.4f}")
print(f"Loss:     {eval_results['eval_loss']:.4f}")

# Detailed classification report
predictions = trainer.predict(val_tok)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

print(f"\nClassification Report:")
print(classification_report(labels, preds, target_names=['Negative', 'Positive']))

🔍 Evaluating model...


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.096918,0.345248,3,0.852000,0.856031



✅ Evaluation Results:
Accuracy: 0.8520
F1 Score: 0.8560
Loss:     0.3452



Classification Report:
              precision    recall  f1-score   support

    Negative       0.89      0.81      0.85       254
    Positive       0.82      0.89      0.86       246

    accuracy                           0.85       500
   macro avg       0.85      0.85      0.85       500
weighted avg       0.85      0.85      0.85       500



In [8]:
# Cell 7 — Test with Custom Sentences
from transformers import pipeline

# Create sentiment pipeline
sentiment_pipeline = pipeline(
    "text-classification",
    model=model_nlp,
    tokenizer=tokenizer,
    device=-1  # CPU
)

# Test with custom sentences
test_sentences = [
    "This movie was absolutely amazing, I loved every minute of it!",
    "Terrible film, complete waste of time and money.",
    "The acting was decent but the plot was confusing.",
    "One of the best movies I have ever seen in my life!",
    "I fell asleep halfway through, so boring.",
    "Brilliant performances by all the cast members!"
]

print("🎭 Sentiment Analysis Results:")
print("="*60)
for sentence in test_sentences:
    result = sentiment_pipeline(sentence)[0]
    label = "😊 POSITIVE" if result['label'] == 'LABEL_1' else "😞 NEGATIVE"
    print(f"\nText: {sentence[:50]}...")
    print(f"Prediction: {label} (confidence: {result['score']:.1%})")

🎭 Sentiment Analysis Results:

Text: This movie was absolutely amazing, I loved every m...
Prediction: 😊 POSITIVE (confidence: 96.0%)

Text: Terrible film, complete waste of time and money....
Prediction: 😞 NEGATIVE (confidence: 94.1%)

Text: The acting was decent but the plot was confusing....
Prediction: 😞 NEGATIVE (confidence: 93.5%)

Text: One of the best movies I have ever seen in my life...
Prediction: 😊 POSITIVE (confidence: 95.8%)

Text: I fell asleep halfway through, so boring....
Prediction: 😞 NEGATIVE (confidence: 90.7%)

Text: Brilliant performances by all the cast members!...
Prediction: 😊 POSITIVE (confidence: 95.9%)


In [9]:
# Cell 8 — Save Model
import os
os.makedirs('../models/saved/bert_sentiment', exist_ok=True)

# Save model and tokenizer
model_nlp.save_pretrained('../models/saved/bert_sentiment')
tokenizer.save_pretrained('../models/saved/bert_sentiment')

# Save results
results = {
    'model': 'DistilBERT',
    'dataset': 'IMDb (2000 train, 500 val)',
    'accuracy': 0.8520,
    'f1_score': 0.8560,
    'loss': 0.3452,
    'epochs': 3
}

import json
with open('../models/saved/bert_sentiment/results.json', 'w') as f:
    json.dump(results, f, indent=4)

print("✅ Model saved successfully!")
print("📁 Saved to: models/saved/bert_sentiment/")
print("\n📊 Final Results:")
for key, value in results.items():
    print(f"  {key}: {value}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved successfully!
📁 Saved to: models/saved/bert_sentiment/

📊 Final Results:
  model: DistilBERT
  dataset: IMDb (2000 train, 500 val)
  accuracy: 0.852
  f1_score: 0.856
  loss: 0.3452
  epochs: 3
